In [6]:
import pytest
from api.silo_service import simulate_silo_response


def print_section_header(title: str):
    """Utility to print prominent, clean section headers."""
    border = "=" * 60
    print(f"\n{border}\n  TEST RUN: {title}\n{border}")


def print_clean_dict(data: dict, indent: int = 2):
    """Recursively formats and prints dictionary structures clearly."""
    spacing = " " * indent
    for key, value in data.items():
        if isinstance(value, dict):
            print(f"{spacing}▸ {key}:")
            print_clean_dict(value, indent + 4)
        elif isinstance(value, list):
            print(f"{spacing}▸ {key}: [List of {len(value)} items]")
            if value and len(value) <= 3:
                for item in value:
                    print(f"{spacing}    • {item}")
        else:
            print(f"{spacing}▸ {key:<18}: {value}")


# ---------------------------------------------------------------------------
# Test 1: The "happy path" — everything normal, expect success.
# ---------------------------------------------------------------------------
def test_ball_beam_simulation_succeeds():
    print_section_header("Ball & Beam Happy Path Simulation")

    request = {
        "system_name": "ball_beam",
        "controller_type": "PID",
        "gains": {"Kp": 4.0, "Ki": 1.5, "Kd": 0.5},
        "dt": 0.01,
        "max_time": 5.0,
    }

    print("\n[INPUT REQUEST]")
    print_clean_dict(request)

    result = simulate_silo_response(request)

    print("\n[SIMULATION OUTPUT]")
    print_clean_dict(result)

    # Assertions
    assert result["success"] is True, f"Simulation should succeed, got: {result.get('message')}"
    assert "metrics" in result
    assert result["metrics"], "metrics dict should not be empty"

    mse = result["metrics"]["mse"]
    assert mse >= 0, f"mse should never be negative, got {mse}"
    assert len(result["trajectory"]) > 0

    print("\n✓ ASSERTIONS PASSED: Output structure and metrics are valid.")


# ---------------------------------------------------------------------------
# Test 2: Failure mode / Graceful fallback handling.
# ---------------------------------------------------------------------------
def test_unknown_system_does_not_crash():
    print_section_header("Unknown System Handling (Fallback Test)")

    request = {
        "system_name": "this_machine_does_not_exist",
        "gains": {"Kp": 1.0},
    }

    print("\n[INPUT REQUEST]")
    print_clean_dict(request)

    result = simulate_silo_response(request)

    print("\n[SIMULATION OUTPUT]")
    print_clean_dict(result)

    assert isinstance(result, dict)
    assert "success" in result

    print("\n✓ ASSERTIONS PASSED: System handled unmapped input gracefully.")


# ---------------------------------------------------------------------------
# Test 3: PARAMETRIZE — run the SAME test across all built-in plants.
# ---------------------------------------------------------------------------
@pytest.mark.parametrize("system_name", ["ball_beam", "dc_motor", "inverted_pendulum"])
def test_all_builtin_plants_simulate(system_name):
    print_section_header(f"Parametrized System Test: {system_name.upper()}")

    request = {
        "system_name": system_name,
        "controller_type": "PID",
        "gains": {"Kp": 4.0, "Ki": 1.5, "Kd": 0.5},
        "dt": 0.01,
        "max_time": 5.0,
    }

    print("\n[INPUT REQUEST]")
    print_clean_dict(request)

    result = simulate_silo_response(request)

    print("\n[SIMULATION OUTPUT]")
    print_clean_dict(result)

    assert result["success"] is True, f"{system_name} failed: {result.get('message')}"
    assert result["metrics"]["mse"] >= 0

    print(f"\n✓ ASSERTIONS PASSED: {system_name} completed simulation and calculated valid metrics.")

In [7]:
test_ball_beam_simulation_succeeds()


  TEST RUN: Ball & Beam Happy Path Simulation

[INPUT REQUEST]
  ▸ system_name       : ball_beam
  ▸ controller_type   : PID
  ▸ gains:
      ▸ Kp                : 4.0
      ▸ Ki                : 1.5
      ▸ Kd                : 0.5
  ▸ dt                : 0.01
  ▸ max_time          : 5.0

[SIMULATION OUTPUT]
  ▸ success           : True
  ▸ metrics:
      ▸ mse               : 0.0016681348869026008
      ▸ rmse              : 0.04084280703995014
      ▸ settling_time     : 2.11
      ▸ overshoot         : 45.83255737807453
      ▸ stable            : True
      ▸ stability_margin  : 94.58354216674022
      ▸ rise_time         : 0.35000000000000003
      ▸ zero_crossings    : 5
      ▸ control_effort    : 0.006749343490671388
      ▸ control_zero_crossings: 8
      ▸ ss_error          : 0.0013781319084091876
  ▸ trajectory: [List of 501 items]
  ▸ control_signals: [List of 501 items]
  ▸ errors: [List of 501 items]
  ▸ initial_state     : None
  ▸ time_points: [List of 501 items]

✓ AS

In [8]:
test_unknown_system_does_not_crash()


  TEST RUN: Unknown System Handling (Fallback Test)

[INPUT REQUEST]
  ▸ system_name       : this_machine_does_not_exist
  ▸ gains:
      ▸ Kp                : 1.0

[SIMULATION OUTPUT]
  ▸ success           : True
  ▸ metrics:
      ▸ mse               : 0.2294137334915716
      ▸ rmse              : 0.47897153724576536
      ▸ settling_time     : inf
      ▸ overshoot         : 113.22458449356598
      ▸ stable            : False
      ▸ stability_margin  : -490.0804530020636
      ▸ rise_time         : 0.59
      ▸ zero_crossings    : 4
      ▸ control_effort    : 0.04315244671325875
      ▸ control_zero_crossings: 4
      ▸ ss_error          : 0.6345619743142207
  ▸ trajectory: [List of 501 items]
  ▸ control_signals: [List of 501 items]
  ▸ errors: [List of 501 items]
  ▸ initial_state     : None
  ▸ time_points: [List of 501 items]

✓ ASSERTIONS PASSED: System handled unmapped input gracefully.


In [11]:
# @pytest.mark.parametrize("system_name", ["ball_beam", "dc_motor", "inverted_pendulum"])

system_name = "inverted_pendulum"
test_all_builtin_plants_simulate(system_name)


  TEST RUN: Parametrized System Test: INVERTED_PENDULUM

[INPUT REQUEST]
  ▸ system_name       : inverted_pendulum
  ▸ controller_type   : PID
  ▸ gains:
      ▸ Kp                : 4.0
      ▸ Ki                : 1.5
      ▸ Kd                : 0.5
  ▸ dt                : 0.01
  ▸ max_time          : 5.0

[SIMULATION OUTPUT]
  ▸ success           : True
  ▸ metrics:
      ▸ mse               : 0.0006950729813212797
      ▸ rmse              : 0.026364236786246623
      ▸ settling_time     : 0.27
      ▸ overshoot         : 3.958282404723608
      ▸ stable            : True
      ▸ stability_margin  : 93.7762503344835
      ▸ rise_time         : 0.27
      ▸ zero_crossings    : 1
      ▸ control_effort    : 0.0011974083737338824
      ▸ control_zero_crossings: 2
      ▸ ss_error          : 0.001902520496832533
  ▸ trajectory: [List of 501 items]
  ▸ control_signals: [List of 501 items]
  ▸ errors: [List of 501 items]
  ▸ initial_state     : None
  ▸ time_points: [List of 501 items]

✓

In [ ]:
{
  "system_name": "ball_beam",
  "gains": {
    "additionalProp1": 5.0,
    "additionalProp2": 0.1,
    "additionalProp3": 1.0
  },
  "controller_type": "PID",
  "scenario": {
    "additionalProp1": {}
  },
  "custom_dynamics_path": "string",
  "dt": 0.01,
  "max_time": 5,
  "target": 0,
  "min_ctrl": -10,
  "max_ctrl": 10
}